In [1]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os

# Import helper modules
from helpers import (
    # Config
    BaseAgentConfig,
    TICKERS,
    MODELS_DIR,
    
    # Training
    train_base_agent,
    evaluate_agent,
    plot_training_history,
    plot_agent_performance,
    
    # Utils
    print_metrics,
    save_agent_results,
    compile_base_agent_results,
)

# Additional imports for diagnostics
from stable_baselines3 import PPO
from helpers.environments import create_env

# Plotting setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✅ Imports complete")
print(f"   Tickers: {TICKERS}")
print(f"   Models directory: {MODELS_DIR}")

✅ Imports complete
   Tickers: ['NVDA', 'MU', 'AAPL', 'AMD', 'ASML', 'MSFT', 'GOOG']
   Models directory: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/harlf_weekly/notebooks/../models


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
print("="*80)
print("AGENT BEHAVIOR DIAGNOSTIC")
print("="*80)

# Try to load a trained model (if it exists)
model_path = "../models/technical_ema_sharpe_softmax"  # Path relative to notebooks directory

print(f"\nLoading model from: {model_path}")

expected_model_file = model_path + '.zip'
if not os.path.isfile(expected_model_file):
    print(f"❌ Model file not found: {expected_model_file}")
    print("Please ensure you have trained the agent first.")
else:
    try:
        model = PPO.load(model_path)
        print("✓ Model loaded successfully")
        
        # Create environment
        print("\nCreating validation environment...")
        env = create_env(
            agent_type='technical',
            split='val',
            reward_type='ema_sharpe',
            normalize_observations=True
        )

        print(f"Environment created: {env}")
        print(f"Action space: {env.action_space}")
        print(f"Observation space: {env.observation_space}")

        # Run one episode and collect actions
        print("\n" + "="*80)
        print("RUNNING EPISODE AND COLLECTING ACTIONS")
        print("="*80)

        obs, _ = env.reset()
        done = False
        actions_taken = []
        positions_held = []
        step_count = 0

        print(f"\n{'Step':<6} {'Action':<60} {'After Constraints':<60}")
        print("-"*130)

        while not done:
            # Get action from agent
            action, _ = model.predict(obs, deterministic=True)
            
            # Format action for display (round to 2 decimals)
            action_str = "[" + " ".join([f"{a:.2f}" for a in action]) + "]"
            print(f"{step_count:<6} {action_str:<60}", end="")
            
            # Take step
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            # Store action and position after environment constraints
            actions_taken.append(action.copy())
            positions_held.append(env.positions.copy())
            
            # Format position for display (round to 2 decimals)
            position_str = "[" + " ".join([f"{p:.2f}" for p in env.positions]) + "]"
            print(f"{position_str:<60}")
            
            step_count += 1

        actions_array = np.array(actions_taken)
        positions_array = np.array(positions_held)

        print(f"\nCompleted {step_count} steps total")

        # Analyze actions
        print("\n" + "="*80)
        print("ACTION ANALYSIS")
        print("="*80)

        print(f"\nTotal steps: {step_count}")
        print(f"Actions shape: {actions_array.shape}")
        print(f"Positions shape: {positions_array.shape}")

        # Check if actions are constant
        print("\n1. ARE ACTIONS CONSTANT?")
        print("-" * 40)
        action_std = actions_array.std(axis=0)
        print(f"Standard deviation per asset (raw actions):")
        for i, ticker in enumerate(TICKERS):
            print(f"  {ticker}: {action_std[i]:.2f}")

        if np.all(action_std < 0.001):
            print("\n⚠️ WARNING: Actions are CONSTANT! Agent is not making real decisions.")
        else:
            print("\n✓ Actions vary over time (agent is making decisions)")

        # Check if positions are constant (after environment constraints)
        print("\n2. ARE POSITIONS CONSTANT?")
        print("-" * 40)
        position_std = positions_array.std(axis=0)
        print(f"Standard deviation per asset (after constraints):")
        for i, ticker in enumerate(TICKERS):
            print(f"  {ticker}: {position_std[i]:.2f}")

        if np.all(position_std < 0.001):
            print("\n🚨 CRITICAL: Positions are CONSTANT! Agent is holding equal weights!")
            print("   This means the agent is NOT actually rebalancing.")
            print("   All returns are from market drift, not active management.")
        else:
            print("\n✓ Positions vary over time (agent is rebalancing)")

        # Check diversity of actions
        print("\n3. ACTION DIVERSITY")
        print("-" * 40)
        unique_actions = len(np.unique(np.round(actions_array, 2), axis=0))
        print(f"Unique action vectors: {unique_actions} out of {step_count}")
        print(f"Diversity ratio: {unique_actions/step_count:.1%}")

        if unique_actions < 5:
            print("\n⚠️ WARNING: Very low action diversity! Agent may be stuck.")
        elif unique_actions < step_count * 0.1:
            print("\n⚠️ WARNING: Low action diversity. Agent is repeating actions.")
        else:
            print(f"\n✓ Good action diversity")

        # Calculate actual rebalancing
        print("\n4. REBALANCING ACTIVITY")
        print("-" * 40)
        position_changes = np.abs(np.diff(positions_array, axis=0)).sum(axis=1)
        total_turnover = position_changes.sum()
        avg_turnover_per_step = position_changes.mean()

        print(f"Total turnover: {total_turnover:.2f}")
        print(f"Average turnover per step: {avg_turnover_per_step:.2f}")
        print(f"Steps with zero turnover: {np.sum(position_changes == 0)} / {step_count-1}")

        if avg_turnover_per_step < 0.01:
            print("\n🚨 CRITICAL: Almost no rebalancing happening!")
            print("   Agent is holding static positions.")
        elif avg_turnover_per_step < 0.05:
            print("\n⚠️ WARNING: Low rebalancing activity.")
        else:
            print("\n✓ Agent is actively rebalancing")

        # Check if agent is just outputting equal weights
        print("\n5. EQUAL WEIGHT CHECK")
        print("-" * 40)
        equal_weight = 1.0 / len(TICKERS)
        mean_weights = positions_array.mean(axis=0)
        print(f"Mean weights:")
        for i, ticker in enumerate(TICKERS):
            diff_from_equal = mean_weights[i] - equal_weight
            print(f"  {ticker}: {mean_weights[i]:.2f} (equal={equal_weight:.2f}, diff={diff_from_equal:+.2f})")

        max_deviation = np.abs(mean_weights - equal_weight).max()
        if max_deviation < 0.01:
            print(f"\n🚨 CRITICAL: Agent is holding equal weights!")
            print(f"   Max deviation from equal weight: {max_deviation:.2f}")
            print(f"   This means the agent learned NOTHING.")
        else:
            print(f"\n✓ Agent has non-equal weights (max deviation: {max_deviation:.2f})")

        # Final diagnosis
        print("\n" + "="*80)
        print("FINAL DIAGNOSIS")
        print("="*80)

        issues = []

        if np.all(action_std < 0.001):
            issues.append("❌ Actions are constant (not making decisions)")
            
        if np.all(position_std < 0.001):
            issues.append("❌ Positions are constant (no rebalancing)")
            
        if unique_actions < step_count * 0.1:
            issues.append("❌ Very low action diversity")
            
        if avg_turnover_per_step < 0.01:
            issues.append("❌ Almost no trading activity")
            
        if max_deviation < 0.01:
            issues.append("❌ Holding equal weights (learned nothing)")

        if issues:
            print("\n🚨 CRITICAL ISSUES FOUND:")
            for issue in issues:
                print(f"   {issue}")
            
            print("\n" + "="*80)
            print("ROOT CAUSE ANALYSIS")
            print("="*80)
            
            print("\nPossible causes:")
            print("1. Agent collapsed to degenerate policy (equal weights)")
            print("2. Environment constraint enforcement too aggressive")
            print("3. Reward function not differentiating between actions")
            print("4. Network too small/large for the problem")
            print("5. Learning rate too high (policy collapsed)")
            print("6. Normalization issues causing constant observations")
            
            print("\nRecommended actions:")
            print("1. Check environment.step() - are constraints overriding agent decisions?")
            print("2. Check reward function - is it providing meaningful gradients?")
            print("3. Try training with different hyperparameters")
            print("4. Verify observations are changing (not constant)")
            print("5. Check if entropy coefficient is too low (not exploring)")
            
        else:
            print("\n✅ Agent appears to be working correctly!")
            print("   - Actions vary over time")
            print("   - Active rebalancing detected")
            print("   - Non-equal weight portfolio")
            print("   - Good action diversity")

        print("\n" + "="*80)
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

AGENT BEHAVIOR DIAGNOSTIC

Loading model from: ../models/technical_ema_sharpe_softmax
✓ Model loaded successfully

Creating validation environment...
Environment created: <PortfolioEnv instance>
Action space: Box(0.0, 1.0, (7,), float32)
Observation space: Box(-inf, inf, (147,), float32)

RUNNING EPISODE AND COLLECTING ACTIONS

Step   Action                                                       After Constraints                                           
----------------------------------------------------------------------------------------------------------------------------------
0      [0.18 0.15 0.14 0.08 0.11 0.14 0.20]                        [0.18 0.15 0.14 0.08 0.11 0.14 0.20]                        
1      [0.18 0.16 0.11 0.09 0.13 0.14 0.20]                        [0.18 0.16 0.11 0.09 0.13 0.14 0.20]                        
2      [0.15 0.13 0.15 0.11 0.16 0.14 0.15]                        [0.15 0.13 0.15 0.11 0.16 0.14 0.15]                        
3      [0.17 0.13 0.20 0.0